# Importing libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import json
import time
import itertools
from sentence_transformers import SentenceTransformer
import os

2025-10-06 13:28:48.345820: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759757328.546157      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759757328.601947      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
if os.environ.get("KAGGLE_URL_BASE", ''):
    print("Running notebook in kaggle.")
    # Installing libraries
    !pip install -U sentence-transformers
else:
    print("Not in kaggle")

in kaggle


In [4]:
# Display all columns
pd.set_option("display.max_columns", None)

# Importing dataset

In [6]:
df = pd.read_parquet('/kaggle/input/cleaned-jobs-and-skills-database-6oct25')
df['job_id'] = df.index
df.head()

,Title,Code,Description,Sector Name,Level,Maximum Notational Hours,Minimum Notational Hours,Version,Originally Approved,Valid Till,Awarding Body,Certifying Bodies,Proposed Occupation,Progression Pathway,Qualifcation Type,Adopted Qualifcation,Training Delivery Hours,NQR code,Qf link,pdf_number,nco_code_full,nco_4digit_code,pc_list,job_id
0,Fire Safety Technician (Oil & Gas),2020/HYC/HSSCI/3611,The main responsibility of the fire safety tec...,Hydrocarbon,Level 4,450 Hours,450 Hours,Version,17 Nov 2022,16 Nov 2025,Hydrocarbon Sector Skill Council (HSSCI),Hydrocarbon Sector Skill Council,"Management of Health, Safety and Environment (...",Senior Fire Safety Technician,General Qualification,N.A.,"{""Theory"":""120"",""Practical"":""240"",""Employabili...",2020/HYC/HSSCI/3611,https://www.nqr.gov.in/qualification/file/STT-...,0,[3119.0800],[3119],[maintain fire safety equipment as per mainten...,0
1,Hindi Typist,2020/OAFM/MEPSC/03792,"The Hindi Typist, is responsible for formattin...",Management,Level 4,450 Hours,390 Hours,Version,17 Nov 2022,17 Nov 2025,Management & Entrepreneurship and Professional...,Management Entrepreneurship and Professional S...,Office Support,Multi-functional Office Executive,General Qualification,N.A.,"{""Theory"":""150"",""Practical"":""180"",""Employabili...",2020/OAFM/MEPSC/03792,https://www.nqr.gov.in/qualification/file/QFil...,1,[4131.9900],[4131],[access specified data or information using sp...,1
2,Certificate Course in Coding Skills,2020/ITES/ASAP/03802,Individuals at this job are responsible for de...,IT-ITeS,Level 5,270 Hours,270 Hours,Version,25 Jun 2020,01 Mar 2026,"Additional Skill Acquisition Programme, Govern...","Additional Skill Acquisition Programme, Govern...",Software Engineer /Project Engineer,"VERTICAL PROGRESSION \nEngineer Trainee, Proje...","Future Skills Qualification,General Qualification",N.A.,"{""Theory"":""36"",""Practical"":""204"",""Employabilit...",2020/ITES/ASAP/03802,https://www.nqr.gov.in/qualification/file/Q%20...,2,[2512.0800],[2512],None,2
3,Transit and Self-Loading Mixer Operator,2020/CON/IESC/3881,Transit and Self-Loading Mixer operator drives...,Infrastructure,Level 4,390 Hours,390 Hours,Version,17 Oct 2019,17 Oct 2022,Infrastructure Equipment Sector Skill Council,Infrastructure Equipment Sector Council,Transit and Self-Loading Mixer Operator,Senior Transit and Self-Loading mixer operator,General Qualification,N.A.,"{""Theory"":""90"",""Practical"":""150"",""Employabilit...",2020/CON/IESC/3881,https://nqr.gov.in/sites/default/files/QF%20-I...,3,[8114.0300],[8114],[Provide basic first aid support and report in...,3
4,AI – Data Architect,2020/ITES/ITSSC/04327,Individuals at this job must be responsible fo...,IT-ITeS,Level 7,750 Hours,660 Hours,Version,19 Dec 2018,22 Sep 2025,IT-ITeS Sector Skills Council NASSCOM (SSC NAS...,IT-ITeS SSC NASSCOM,Artificial Intelligence and Big Data Analytics,"Solutions Architect, Senior Database Administr...",Upskilling Qualification,N.A.,"{""Theory"":""180"",""Practical"":""330"",""Employabili...",2020/ITES/ITSSC/04327,https://www.nqr.gov.in/qualification/file/SSC%...,4,None,None,[encourage team members with diverse view poin...,4


# Preparing dataset

In [7]:
df_exploded = df.explode(column = 'pc_list', ignore_index = True)

In [9]:
df_agg = df_exploded.groupby('pc_list').agg({
    "Sector Name": lambda x: list(set(x)),
    "Level": lambda x: list(set(x)),
    "Title": lambda x: list(set(x)),
    "job_id": lambda x: list(set(x)),
    'pdf_number': lambda x: list(set(x)),
    "nco_4digit_code": lambda x: list({
    v
    for arr in x
    for v in ((arr.tolist() if arr is not None and len(arr) > 0 else [None]))})
}).reset_index()

In [12]:
# Generating skill embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
df_agg['embedding'] = model.encode(df_agg['pc_list'].tolist()).tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2788 [00:00<?, ?it/s]

# Exporting

In [13]:
df_agg.to_pickle("pc_embeddings.pkl")